# IBR 动态建模与仿真

九个教学单元。求解器只依赖 Python 标准库；不需要安装 ibrsim。低频实验使用额定频率代数网络与理想电流 / 电压实现，不等同于完整阶次模型。每次运行保留参数、端口、残差及适用范围。网页版本提供交互图。


In [ ]:
"""IBR course teaching models; Python standard library only.

Adapted conventions/control laws from PINN-IBR, reviewed 2026-10-03.
These are independently authored teaching realizations, not ibrsim models.
Network: balanced nominal-frequency algebraic impedances. GFL: ideal current
actuator + SRF PLL (4 states); droop: ideal voltage source + PQ filters (3);
VSM: adds rotor speed (4). Parallel: both branches share one PCC (7).
Switch: explicit GFL -> VSM reset, with voltage and frequency matching.
LCL: separate open-loop averaged six-state electrical plant in seconds.
No switching ripple, DC dynamics, saturation, faults, or protection.
"""
import math
import cmath

WB = 2 * math.pi * 50
NAMES = {"gfl": ["delta", "xi_pll", "id", "iq"],
         "droop": ["delta", "Pf", "Qf"],
         "vsm": ["delta", "omega", "Pf", "Qf"]}


def bounded(value, low, high, name):
    value = float(value)
    if not math.isfinite(value) or not low <= value <= high:
        raise ValueError(f"{name} must be finite and in [{low}, {high}]")
    return value


def settings(case):
    mode = case.get("mode", "droop")
    if mode not in ("frame", "lcl", "gfl", "droop", "vsm", "parallel", "switch", "compare"):
        raise ValueError("Unknown model mode")
    event = case.get("event", "p")
    if event not in ("p", "q", "v", "f"):
        raise ValueError("Unknown disturbance")
    limit = .01 if mode == "lcl" else .5 if event == "f" else .08
    return dict(mode=mode, event=event,
                step=bounded(case.get("step", .005 if mode == "lcl" else .03), -limit, limit, "step"),
                scr=bounded(case.get("scr", 5), 2, 10, "SCR"),
                mp=bounded(case.get("mp", .02), .005, .04, "m_p"),
                inertia=bounded(case.get("inertia", 4), .5, 8, "M"),
                bandwidth=bounded(case.get("bandwidth", 2), .5, 5, "PLL bandwidth"),
                angle=bounded(case.get("angle", 20), -180, 180, "frame angle"),
                dt=bounded(case.get("dt", .0005), .0001, .001, "time step"),
                duration=bounded(case.get("duration", 4), 4, 20, "duration"))


def rk4(rhs, x, h):
    a = rhs(x)
    b = rhs([v + h * d / 2 for v, d in zip(x, a)])
    c = rhs([v + h * d / 2 for v, d in zip(x, b)])
    d = rhs([v + h * e for v, e in zip(x, c)])
    return [v + h * (aa + 2 * bb + 2 * cc + dd) / 6
            for v, aa, bb, cc, dd in zip(x, a, b, c, d)]


def high_voltage_power_flow(s, z):
    """V = 1 + Z conj(S/V), choosing the high-voltage root."""
    a, b = (z * s.conjugate()).real, (z * s.conjugate()).imag
    discriminant = 1 + 4 * a - 4 * b * b
    if discriminant <= 0:
        raise ValueError("No qualified high-voltage equilibrium")
    magnitude_squared = (1 + 2 * a + math.sqrt(discriminant)) / 2
    return complex(magnitude_squared - a, b)


class TeachingCase:
    def __init__(self, kind, cfg):
        self.cfg = cfg
        self.kinds = ["gfl", "droop"] if kind == "parallel" else ["gfl" if kind == "switch" else kind]
        self.zg = complex(1, 10) / (cfg["scr"] * math.sqrt(101))
        self.zf = complex(.00625, .1)
        self.e0, self.commands, self.slices, self.x0 = [], [], [], []
        voltage = high_voltage_power_flow(complex(.6, 0), self.zg)
        current = complex(.6 / len(self.kinds), 0).conjugate() / voltage.conjugate()
        for family in self.kinds:
            start = len(self.x0)
            if family == "gfl":
                local = current * cmath.exp(-1j * cmath.phase(voltage))
                state = [cmath.phase(voltage), 0, local.real, local.imag]
                self.e0.append(1.)
                self.commands.append([.6 / len(self.kinds), 0.])
            else:
                source = voltage + self.zf * current
                state = [cmath.phase(source), .6 / len(self.kinds), 0]
                if family == "vsm":
                    state.insert(1, 1.)
                self.e0.append(1.)
                self.commands.append([.6 / len(self.kinds), (abs(source) - 1) / .0325])
            self.x0.extend(state)
            self.slices.append((start, len(self.x0)))

    def evaluate(self, x, after=False):
        cfg = self.cfg
        step = cfg["step"] if after else 0
        voltage_grid = 1 + (step if cfg["event"] == "v" else 0)
        omega_grid = 1 + (step / 50 if cfg["event"] == "f" else 0)
        dp = step / len(self.kinds) if cfg["event"] == "p" else 0
        dq = step / len(self.kinds) if cfg["event"] == "q" else 0
        forcing, denominator, entries = complex(voltage_grid), complex(1), []
        for j, (family, (start, stop)) in enumerate(zip(self.kinds, self.slices)):
            xb = x[start:stop]
            if family == "gfl":
                value = complex(xb[2], xb[3]) * cmath.exp(1j * xb[0])
                forcing += self.zg * value
            else:
                qf = xb[3] if family == "vsm" else xb[2]
                amplitude = self.e0[j] - .0325 * (qf - self.commands[j][1] - dq)
                if not .5 < amplitude < 1.5:
                    raise ValueError("Voltage source left the teaching model domain")
                value = amplitude * cmath.exp(1j * xb[0])
                forcing += self.zg * value / self.zf
                denominator += self.zg / self.zf
            entries.append(value)
        voltage = forcing / denominator
        if not .5 < abs(voltage) < 1.5:
            raise ValueError("PCC voltage left the teaching model domain")
        derivatives, currents, powers, omegas = [], [], [], []
        for j, (family, (start, stop), value) in enumerate(zip(self.kinds, self.slices, entries)):
            xb = x[start:stop]
            current = value if family == "gfl" else (value - voltage) / self.zf
            s = voltage * current.conjugate()
            p_ref, q_ref = self.commands[j][0] + dp, self.commands[j][1] + dq
            if family == "gfl":
                local = voltage * cmath.exp(-1j * xb[0])
                error = local.imag / abs(voltage)
                wn = 2 * math.pi * cfg["bandwidth"]
                omega = 1 + 2 * .707 * wn / WB * error + wn * wn / WB * xb[1]
                ref = complex(p_ref, -q_ref) / local.conjugate()
                derivative = [WB * (omega - omega_grid), error,
                              (ref.real - xb[2]) / .02, (ref.imag - xb[3]) / .02]
            elif family == "droop":
                omega = 1 - cfg["mp"] * (xb[1] - p_ref)
                derivative = [WB * (omega - omega_grid), (s.real - xb[1]) / .1, (s.imag - xb[2]) / .05]
            else:
                omega = xb[1]
                derivative = [WB * (omega - omega_grid),
                              (p_ref - xb[2] - (omega - 1) / cfg["mp"]) / cfg["inertia"],
                              (s.real - xb[2]) / .1, (s.imag - xb[3]) / .05]
            derivatives.extend(derivative)
            currents.append(current)
            powers.append(s)
            omegas.append(omega)
        current = sum(currents)
        total = voltage * current.conjugate()
        residual = abs(voltage - voltage_grid - self.zg * current)
        return dict(dx=derivatives, voltage=voltage, currents=currents, powers=powers,
                    p=total.real, q=total.imag, v=abs(voltage), frequency=50 * omegas[0],
                    residual=residual, omega=omegas[0])

    def reset_to_vsm(self, x):
        before = self.evaluate(x, True)
        source = before["voltage"] + self.zf * before["currents"][0]
        s, omega = before["powers"][0], before["omega"]
        dp = self.cfg["step"] if self.cfg["event"] == "p" else 0
        dq = self.cfg["step"] if self.cfg["event"] == "q" else 0
        self.commands[0][0] = s.real + (omega - 1) / self.cfg["mp"] - dp
        self.e0[0] = abs(source) + .0325 * (s.imag - self.commands[0][1] - dq)
        self.kinds[0] = "vsm"
        reset = [cmath.phase(source), omega, s.real, s.imag]
        after = self.evaluate(reset, True)
        error = max(abs(after["voltage"] - before["voltage"]), abs(after["p"] - before["p"]),
                    abs(after["q"] - before["q"]), abs(after["omega"] - omega))
        return reset, error


def controller_run(kind, cfg):
    model = TeachingCase(kind, cfg)
    x = model.x0[:]
    initial_residual = max(map(abs, model.evaluate(x)["dx"]))
    time, traces = [], {key: [] for key in ("p", "q", "v", "frequency")}
    if kind == "parallel":
        traces.update(p_gfl=[], p_gfm=[])
    network_residual, reset_error = 0., None
    # Events land exactly on step boundaries; each segment has frozen inputs.
    segments = [(0., 1., False), (1., 2. if kind == "switch" else cfg["duration"], True)]
    if kind == "switch":
        segments.append((2., cfg["duration"], True))
    for segment, (start, stop, after) in enumerate(segments):
        if kind == "switch" and segment == 2:
            x, reset_error = model.reset_to_vsm(x)
        count = math.ceil((stop - start) / cfg["dt"])
        h = (stop - start) / count
        stride = max(1, round(.01 / h))
        for k in range(count + 1):
            if k % stride == 0 or k == count:
                e = model.evaluate(x, after)
                network_residual = max(network_residual, e["residual"])
                time.append(stop if k == count else start + k * h)
                for key in ("p", "q", "v", "frequency"):
                    traces[key].append(e[key])
                if kind == "parallel":
                    traces["p_gfl"].append(e["powers"][0].real)
                    traces["p_gfm"].append(e["powers"][1].real)
            if k < count:
                x = rk4(lambda state: model.evaluate(state, after)["dx"], x, h)
    manifest = [f"{family}.{name}" for family in model.kinds for name in NAMES[family]]
    return dict(time=time, traces=traces, mode=kind, state_count=len(x), state_names=manifest,
                initial_residual=initial_residual, network_residual=network_residual,
                reset_error=reset_error, final_commands=model.commands,
                scope="nominal-frequency algebraic network; ideal current/voltage realization")


def frame_run(cfg):
    offset = cfg["angle"] * math.pi / 180
    time = [k / 10000 for k in range(401)]
    traces = {key: [] for key in ("va", "vb", "vc", "vd", "vq", "p", "q")}
    residual = 0.
    for t in time:
        phases = [WB * t + shift for shift in (0, -2 * math.pi / 3, 2 * math.pi / 3)]
        va = [math.sqrt(2 / 3) * math.cos(p) for p in phases]
        ia = [math.sqrt(2 / 3) * .6 * math.cos(p - math.pi / 6) for p in phases]
        vd = sum(math.sqrt(2 / 3) * math.cos(p + offset) * a for p, a in zip(phases, va))
        vq = sum(-math.sqrt(2 / 3) * math.sin(p + offset) * a for p, a in zip(phases, va))
        id_ = sum(math.sqrt(2 / 3) * math.cos(p + offset) * a for p, a in zip(phases, ia))
        iq = sum(-math.sqrt(2 / 3) * math.sin(p + offset) * a for p, a in zip(phases, ia))
        p, q = vd * id_ + vq * iq, vq * id_ - vd * iq
        residual = max(residual, abs(p - sum(v * i for v, i in zip(va, ia))))
        for key, value in zip(traces, [*va, vd, vq, p, q]):
            traces[key].append(value)
    return dict(time=time, traces=traces, mode="frame", state_count=0, initial_residual=residual,
                network_residual=0., reset_error=None, scope="balanced power-invariant Park transform")


def lcl_run(cfg):
    r1, ell1, cf = .00625, 8.4375e-5, .0008
    zg = complex(1, 10) / (cfg["scr"] * math.sqrt(101))
    r2, ell2 = .00625 + zg.real, (.1 + zg.imag) / WB
    i2 = complex(.6)
    vc = 1 + complex(r2, WB * ell2) * i2
    i1 = i2 + 1j * WB * cf * vc
    u = vc + complex(r1, WB * ell1) * i1
    x = [i1.real, i1.imag, vc.real, vc.imag, i2.real, i2.imag]
    def evaluate(state, after):
        a, v, b = complex(*state[:2]), complex(*state[2:4]), complex(*state[4:])
        command = u + (cfg["step"] if after else 0)
        da = (command - v - r1 * a) / ell1 - 1j * WB * a
        dv = (a - b) / cf - 1j * WB * v
        db = (v - 1 - r2 * b) / ell2 - 1j * WB * b
        denergy = ell1 * (a.conjugate() * da).real + cf * (v.conjugate() * dv).real + ell2 * (b.conjugate() * db).real
        balance = (command * a.conjugate()).real - b.real - r1 * abs(a)**2 - r2 * abs(b)**2
        return [da.real, da.imag, dv.real, dv.imag, db.real, db.imag], abs(denergy - balance)
    initial = max(map(abs, evaluate(x, False)[0]))
    time, traces, residual = [], {key: [] for key in ("i2d", "vcd", "vcq")}, 0.
    for start, stop, after in [(0., .01, False), (.01, .03, True)]:
        count = round((stop - start) / .000002)
        h = (stop - start) / count
        for k in range(count + 1):
            if k % 25 == 0 or k == count:
                time.append(stop if k == count else start + k * h)
                for key, value in zip(traces, [x[4], x[2], x[3]]):
                    traces[key].append(value)
                residual = max(residual, evaluate(x, after)[1])
            if k < count:
                x = rk4(lambda state: evaluate(state, after)[0], x, h)
    return dict(time=time, traces=traces, mode="lcl", state_count=6, initial_residual=initial,
                network_residual=residual, reset_error=None, scope="open-loop six-state averaged LCL; energy balance audit")


def solve(case):
    cfg = settings(case)
    if cfg["mode"] == "frame":
        result = frame_run(cfg)
    elif cfg["mode"] == "lcl":
        result = lcl_run(cfg)
    elif cfg["mode"] == "compare":
        runs = {kind: controller_run(kind, cfg) for kind in ("gfl", "droop", "vsm", "parallel")}
        first = runs["gfl"]
        result = dict(time=first["time"], traces={kind: run["traces"]["p"] for kind, run in runs.items()},
                      mode="compare", state_count=0, initial_residual=max(r["initial_residual"] for r in runs.values()),
                      network_residual=max(r["network_residual"] for r in runs.values()), reset_error=None,
                      runs={kind: {k: v for k, v in run.items() if k not in ("time", "traces")} for kind, run in runs.items()},
                      scope=first["scope"])
    else:
        result = controller_run(cfg["mode"], cfg)
    result["case"] = cfg
    return result



## C1-01 · 系统边界与模型表示

<h3>先确定问题和端口</h3>
<p>逆变器型资源通过受控变流器连接电网。模型边界可以设在变流器平均电压端、滤波电容端、连接点（PoC），或整个场站母线。电容端测得的功率与 PoC 功率不同，因为两者之间的支路会储能和消耗功率。</p>
<p>本课程从这条链路开始：<strong>变流器 → L₁ → 电容 C → L₂ → PoC → 电网阻抗 → 无穷大母线</strong>。无穷大母线给定电压幅值和频率；直流供电理想化。正 P、Q 表示从声明的测量端口向网络注入。控制器接收指令和测量，并不能直接指定所有电气状态。</p>
<h3>同一设备的三种表示</h3>
<div class="ibr-table"><table><thead><tr><th>表示</th><th>保留内容</th><th>可研究的问题</th></tr></thead><tbody><tr><td>开关模型</td><td>具体开关动作与对应的电气变化</td><td>某个开关策略产生多大的纹波？</td></tr><tr><td>平均 dq 模型</td><td>平均电压指令、物理电路和控制器动态</td><td>电流环如何与 LCL 动态相互作用？</td></tr><tr><td>正序 / 低频模型</td><td>选定的包络、同步和功率动态</td><td>小幅功率指令如何影响同步？</td></tr></tbody></table></div>
<p>模型表示与控制类别是两个独立选择。GFL 可以连接显式 LCL 电路，也可以用理想电流执行环节表示。低阶 GFM 电压源可以省略内部电压环和电流环；省略之后，模型能够回答的问题也改变了。</p>
<h3>先写接口，再写方程</h3>
<p>x 表示微分状态，z 表示代数网络变量，u 表示指令和电网条件，y 表示输出。网络约束必须与控制器一致求解。有储能或积分演化规律的量是状态；通过阻抗关系计算出的电流则是代数量。</p>
<p><strong>分类示例：</strong>本课程 droop 实验中，x = [δ, P_f, Q_f]。PCC 电压与支路电流属于 z；P*、Q*、V_g、ω_g 属于输入。电容电压不在状态中，因此这个实验不能展示电容谐振。模块 3 会单独保留它。</p>
<h3>第一个实验说明什么</h3>
<p>下方四条曲线都从 PoC 总 P = 0.6 pu、Q = 0 出发，在 1 s 增加 0.03 pu 指令。保留的控制动态不同，轨迹也不同。终点接近不能证明模型可互换。每次展示结果，都应同时保留状态清单、测量端口、参数和扰动。</p>

$$
\dot{x}=f(x,z,u),\qquad 0=g(x,z,u),\qquad y=h(x,z,u)
$$

1. 运行默认比较，指出每条曲线保留了哪些动态。
2. 改为电压阶跃，解释为什么这些轨迹不能检验开关纹波精度。
3. 写出四行模型约定：边界、状态、输入、测量输出。


In [ ]:
case = {"mode": "compare", "scr": 5, "step": 0.03, "event": "p", "duration": 4}
result = solve(case)
print("states:", result["state_count"])
print("initial residual:", result["initial_residual"])
print("network / energy residual:", result["network_residual"])
print("reset:", result["reset_error"])
print("final:", {k: values[-1] for k, values in result["traces"].items()})


在三状态 droop 实验中，哪个量是代数量？

1. 滤波有功功率
2. PCC 电压
3. 内部角度

<details><summary>解释</summary>

PCC 电压由网络约束得到，P_f 和 δ 有微分方程。如果显式保留电容，该电容电压就是状态；分类取决于模型表示。

</details>


## C1-02 · 坐标变换与标幺约定

<h3>坐标旋转不改变物理功率</h3>
<p>对无零序的平衡三相信号，功率不变 Park 矩阵将 abc 映射到两个旋转轴。两行分别使用 cos(θ + φₖ) 和 −sin(θ + φₖ)，φₖ = [0, −2π/3, 2π/3]。√(2/3) 的缩放保持三相内积不变。不要将这个变换与幅值不变约定中的 3/2 功率系数混用。</p>
<p>在平衡子空间上，逆变换就是 Park 矩阵的转置。有零序时需要增加一个轴；两轴逆变换无法恢复零序分量。</p>
<h3>功率与电流指令</h3>
<p>电流以流向网络为正：P = v_d i_d + v_q i_q，Q = v_q i_d − v_d i_q。d 轴与电压对齐时 v_q = 0，正 Q 对应负 i_q。下面的电流指令矩阵是这个 P–Q 映射在非零电压下的精确逆映射。</p>
<h3>标幺量背后仍有单位</h3>
<p>选 S_b = 10 kVA、V_b = 400 V 线电压 RMS、f_b = 50 Hz，可得 Z_b = 16 Ω、ω_b = 314.159 rad/s。功率不变 dq 电压基值为 400 V，dq 电流基值为 25 A。物理线电流 RMS 基值为 14.434 A，两者是不同的量。</p>
<p>定义 r = R/Z_b，ℓ = L/Z_b（秒），c = C Z_b（秒），于是额定频率下的电抗 x = ω_b ℓ。若“标幺电感”实际指电抗 x，就不能把它直接代入含时间导数的 ℓ 位置。</p>
<h3>数值检查</h3>
<p>实验电压幅值为 1、电流幅值为 0.6，电流滞后 30°。坐标偏差为零时，v_d = 1、v_q = 0、P = 0.6 cos(30°) = 0.519615、Q = 0.3。坐标偏差 +20° 时，v_d = cos(20°)、v_q = −sin(20°)。电压和电流一起旋转，所以 P、Q 不变。</p>
<p>相对于电网的角度满足 δ̇ = ω_b(ω_pu − ω_g,pu)。在 50 Hz 下，0.001 pu 频差对应 0.314159 rad/s，而不是 0.001 rad/s。</p>

$$
T(\theta)=\sqrt{\frac{2}{3}}\begin{bmatrix}\cos\theta&\cos(\theta-2\pi/3)&\cos(\theta+2\pi/3)\\-\sin\theta&-\sin(\theta-2\pi/3)&-\sin(\theta+2\pi/3)\end{bmatrix}
$$

$$
P=v_d i_d+v_q i_q,\qquad Q=v_q i_d-v_d i_q
$$

$$
\begin{bmatrix}i_d^*\\i_q^*\end{bmatrix}=\frac{1}{v_d^2+v_q^2}\begin{bmatrix}v_d&v_q\\v_q&-v_d\end{bmatrix}\begin{bmatrix}P^*\\Q^*\end{bmatrix}
$$

$$
Z_b=\frac{V_b^2}{S_b},\quad I_{b,dq}=\frac{S_b}{V_b},\quad \dot\delta=\omega_b(\omega_{pu}-\omega_{g,pu})
$$

1. 将坐标偏差设为 0°、+20°、−90°，比较 v_d、v_q。
2. 选择 P–Q 曲线，验证物理功率保持不变。
3. 在 frame_run() 中修改电流相角，先预测 P、Q 的符号，再运行。


In [ ]:
case = {"mode": "frame", "scr": 5, "step": 0.03, "event": "p", "duration": 4}
result = solve(case)
print("states:", result["state_count"])
print("initial residual:", result["initial_residual"])
print("network / energy residual:", result["network_residual"])
print("reset:", result["reset_error"])
print("final:", {k: values[-1] for k, values in result["traces"].items()})


v_d = 1、v_q = 0 时，哪个电流注入 Q = +0.2 pu？

1. i_q = +0.2 pu
2. i_q = 0.3 pu
3. i_q = −0.2 pu

<details><summary>解释</summary>

在这个对齐坐标中 Q = −v_d i_q，因此 i_q = −0.2。答案依赖已声明的坐标变换和注入方向；改变任一约定，都需要同步调整整套方程。

</details>


## C1-03 · 平均变流器与 LCL 电路

<h3>保留六个电气状态</h3>
<p>平均变流器施加连续电压指令 u，不描述逐次开关动作。LCL 电路保留变流器侧电流 i₁、电容电压 v_c、输出电流 i₂，每个量都有 d、q 两个分量。网侧滤波电感与电网电感流过同一电流，所以微分方程可以合并串联 r、ℓ；需要独立端口测量时，再重构 PoC 电压。</p>
<p>取 J = [[0, 1], [−1, 0]]。在以 ω_b ω 旋转的坐标中，复数 dq 向量产生 −jω_bω 倍自身的项；实数向量写法则是 +ω_bωJ 倍自身。这些符号来自旋转坐标变换的求导。</p>
<h3>控制指令不等于闭环状态</h3>
<p>实验先保持平衡点变流器电压，在 10 ms 给 d 分量增加 0.005 pu。这是<strong>开环电路实验</strong>。闭环 droop GFM 还需要角度、P–Q 滤波、电压 PI 和电流 PI 状态。六个电气状态，加一个角度、两个功率滤波状态、四个 PI 积分状态，组成仓库中的 13 状态 droop 实现。</p>
<h3>一致地初始化每个元件</h3>
<p>令无穷大母线端 i₂ = 0.6 − j0，依次计算 v_c = 1 + (r₂ + jω_bℓ₂)i₂、i₁ = i₂ + jω_b c v_c、u = v_c + (r₁ + jω_bℓ₁)i₁。若简单地将电容设为 1、所有电流设为零，会引入并未要求的上电暂态。</p>
<p>默认系数：r₁ = 0.00625、ℓ₁ = 84.375 μs、c = 0.0008 s、r₂ = 0.00625 + r_g、ω_bℓ₂ = 0.1 + x_g。仿真覆盖 30 ms，积分步长为 2 μs。这是明确声明的教学参数，并非对源 Notebook 各参数配置的逐一复现。</p>
<h3>同时检查能量关系</h3>
<p>定义 W = (ℓ₁|i₁|² + c|v_c|² + ℓ₂|i₂|²)/2。dq 交叉项在 Ẇ 中抵消。下方能量变化率恒等式同时检查电压符号、支路方向和损耗项。残差反映方程一致性，不能替代变流器的外部验证。</p>

$$
\dot i_1=\frac{u-v_c-r_1i_1}{\ell_1}+\omega_b\omega Ji_1
$$

$$
\dot v_c=\frac{i_1-i_2}{c}+\omega_b\omega Jv_c,\qquad\dot i_2=\frac{v_c-v_g-r_2i_2}{\ell_2}+\omega_b\omega Ji_2
$$

$$
v_{PoC}=v_c-r_{f2}i_2-\ell_{f2}(\dot i_2-\omega_b\omega Ji_2)
$$

$$
\dot W=u^Ti_1-v_g^Ti_2-r_1\lVert i_1\rVert^2-r_2\lVert i_2\rVert^2
$$

1. 运行 +0.005 pu 变流器电压阶跃，观察事件后 20 ms。
2. 将 SCR 从 5 降为 2 再运行，说明电气支路中哪个参数改变了。
3. 检查 lcl_run()，识别每项储能，并验证能量变化率残差。


In [ ]:
case = {"mode": "lcl", "scr": 5, "step": 0.005, "event": "p", "duration": 4}
result = solve(case)
print("states:", result["state_count"])
print("initial residual:", result["initial_residual"])
print("network / energy residual:", result["network_residual"])
print("reset:", result["reset_error"])
print("final:", {k: values[-1] for k, values in result["traces"].items()})


这个开环 LCL 实验包含哪些状态？

1. 两个电流、一个电压，各有 d、q 分量
2. 六个控制积分状态
3. 只有有功与无功功率

<details><summary>解释</summary>

六个状态是 i₁d、i₁q、v_cd、v_cq、i₂d、i₂q。功率由电气变量计算；加入显式测量滤波器后，滤波功率才成为独立状态。

</details>


## C1-04 · PLL 与跟网型控制

<h3>先跟踪电压角度，再控制电流</h3>
<p>跟网型控制器用锁相环估计网络角度，再将电压、电流表达在估计坐标中。SRF PLL 驱动局部 v_q 趋近零。v_q 为正，表示电压向量领先估计的 d 轴；按这里的符号约定，PLL 应增大估计角度。</p>
<p>四状态教学实现保留 δ、PLL 积分 ξ，以及两个理想电流执行状态 i_d、i_q。20 ms 的电流执行时间常数代替详细电流控制器和 LCL 电路。它适合学习同步与指令跟踪，不能检验内部控制谐振或限流行为。</p>
<h3>归一化 PLL，并保持正确时间尺度</h3>
<p>相位检测误差 e = v_q/|V|。PI 输出为标幺频率偏差：ω̂ = 1 + k_p e + k_i ξ，ξ̇ = e；相对角度满足 δ̇ = ω_b(ω̂ − ω_g)。在刚性电压、接近对齐的条件下，特征多项式为 s² + ω_b k_p s + ω_b k_i。</p>
<p>令 k_p = 2ζω_n/ω_b、k_i = ω_n²/ω_b，其中 ζ = 0.707、ω_n = 2πb。实验的“PLL 带宽” b 是以 Hz 表示的自然频率设计参数，不等于任意网络下实测的闭环 −3 dB 带宽。</p>
<h3>在测量端口将功率映射为电流</h3>
<p>使用模块 2 的完整 P–Q 逆映射，不要在整个暂态中都假设 v_q = 0、仅用 v_d 除指令。局部电流旋转到电网坐标后，网络给出 V = V_g + Z_g I。这个反馈使电压角度与 PLL 动态受到电网阻抗影响。</p>
<p>基准运行点为 PoC 总 P = 0.6、Q = 0。有功指令阶跃改变电流需求，电网频率阶跃使 PLL 跟踪新频率。教学 GFL 未加入 frequency–watt、volt–var 或 RoCoF 注入；它们在仓库高阶 GFL 模型中属于另外的控制环节。</p>
<h3>如何解读弱网实验</h3>
<p>保持指令和增益，降低 SCR，同时比较 P、Q、PCC 幅值和估计频率。曲线平滑只说明这个理想执行模型在声明的测试中有界。仓库中的完整 15 状态 GFL 还保留测量滤波和物理电路状态。</p><h3>接回详细电流控制与物理电路</h3><p>仓库 LCL GFL 先由 PoC 的 P–Q 指令产生网侧电流参考 i₂*。启用电容电流补偿时，i₁* = i₂* − ω_b ω̂ c Jv_c。定义 e_i = i₁* − i₁，并积分 ξ̇_i = e_i；电流 PI、电压前馈和电感解耦生成平均变流器电压 u，再由六个 LCL 方程决定真实电流。完整 15 状态实现保留一个相对角度、一个 PLL 积分、两个电压测量状态、一个频率测量状态、两个功率指令滤波状态、两个电流 PI 积分和六个电气状态。网页四状态模型则是另一种理想执行表示。</p>

$$
e=\frac{v_q}{\lVert v\rVert},\quad\dot\xi=e,\quad\hat\omega=1+k_pe+k_i\xi,\quad\dot\delta=\omega_b(\hat\omega-\omega_g)
$$

$$
k_p=\frac{2\zeta\omega_n}{\omega_b},\qquad k_i=\frac{\omega_n^2}{\omega_b}
$$

$$
\tau_i\dot i_{dq}=i_{dq}^*-i_{dq},\qquad V=V_g+Z_gI
$$

$$
i_1^*=i_2^*-\omega_b\hat\omega cJv_c,\quad\dot\xi_i=i_1^*-i_1
$$

$$
u=v_c-\omega_b\hat\omega\ell_1Ji_1+K_{pi}(i_1^*-i_1)+K_{ii}\xi_i
$$

1. 运行基准有功阶跃，再选择估计频率与 PCC 幅值。
2. 施加 +0.1 Hz 电网频率阶跃，检查暂态后的 PLL 频率。
3. 在 SCR = 2 时，用相同扰动比较 1 Hz 和 5 Hz 的 PLL 设计值。


In [ ]:
case = {"mode": "gfl", "scr": 5, "step": 0.03, "event": "p", "duration": 4}
result = solve(case)
print("states:", result["state_count"])
print("initial residual:", result["initial_residual"])
print("network / energy residual:", result["network_residual"])
print("reset:", result["reset_error"])
print("final:", {k: values[-1] for k, values in result["traces"].items()})


局部 v_q 为正时，哪种 PLL 初始动作符合这里的约定？

1. 降低估计角速度
2. 提高估计角速度
3. 自动将 P* 设为零

<details><summary>解释</summary>

k_p 为正时，正 v_q 产生正频率修正，使估计 d 轴向电压向量靠近。这个 PLL 方程本身不会改变功率指令。

</details>


## C1-05 · 下垂构网型控制

<h3>由功率–频率规律生成角度</h3>
<p>Droop GFM 通过积分频率指令生成内部电压角度，这个同步规律不需要 PLL。滤波有功偏差改变频率，滤波无功偏差改变电压幅值，再由网络决定实际注入功率。</p>
<p>教学模型的三个状态为 δ、P_f、Q_f。内部电压 E exp(jδ) 通过 Z_f = 0.00625 + j0.1 pu 接入网络，支路电流为 (E exp(jδ) − V)/Z_f。物理 LCL 与 PI 环节被这条代数关系替代；Z_f 是教学模型中声明的源阻抗。</p>
<h3>通过网络闭合反馈环</h3>
<p>P* 小幅增加时，P_f 不能瞬间跳变，因此频率首先提高；δ 前移，电气功率增加，随后 P_f 跟上。在额定频率无穷大母线约束下重新达到平衡时，ω = 1、P_f = P*。功率参考是控制输入，不是对测量功率轨迹的直接赋值。</p>
<p>增大 m_p 同时改变稳态频率–功率斜率和暂态环路增益。测量滤波固定为 τ_p = 0.1 s、τ_q = 0.05 s。解释响应时，还要考虑网络和滤波时间尺度。</p>
<h3>匹配端口运行点</h3>
<p>所有控制类别都初始化到 PoC P = 0.6、Q = 0。由于源阻抗中有电流，所需内部电压幅值通常大于 1。E₀ 固定为 1 时，必须选择适当 Q*，使 E = E₀ − n_q(Q_f − Q*) 满足所需幅值。因此端口 Q 为零，并不要求 droop 的 Q* 为零。</p>
<p>SCR = 5 时，基准 Q* 约为 0.316795 pu。它是匹配平衡点的控制偏置，不表示 PCC 注入了同样大小的无功。实验会展示实际指令，便于检查这个区别。</p>
<h3>用频率阶跃检验稳态斜率</h3>
<p>电网频率增加 +0.1 Hz。稳态锁定时 ω = ω_g，P ≈ P* − (ω_g − 1)/m_p。m_p = 0.02 时，预测有功变化为 −0.1/(50 × 0.02) = −0.1 pu。这是模型内部的稳态预测；代数阻抗仍在额定频率下计算。</p><h3>建立完整串级控制实现</h3><p>保留内部环节时，使用 x = [δ, P_f, Q_f, ξ_vd, ξ_vq, ξ_id, ξ_iq, i₁d, i₁q, v_cd, v_cq, i₂d, i₂q]。设 v* = [E, 0]、电压误差 e_v = v* − v_c、ξ̇_v = e_v。启用输出电流前馈和电容解耦时，电压 PI 按下方新增的第一个方程产生 i₁*。电流误差为 e_i = i₁* − i₁、ξ̇_i = e_i；电流 PI 按第二个方程产生变流器电压 u，再代入模块 3 的 LCL 方程，闭合整个模型。</p><p>源仓库经典 droop 实现用 v_c、i₂ 的功率驱动滤波器，基准输出则在 PoC 重构，这两个端口需要区分。初始化无限幅平衡点时，先求电气状态，再令 ξ_v = (i₁ − F i₂ + ω_b ω c Jv_c)/K_iv、ξ_i = (u − v_c + ω_b ω ℓ₁ Ji₁)/K_ii。控制误差为零，不代表 PI 积分状态为零。网页 droop 实验明确省略了这四个积分状态与六个电气状态。</p>

$$
\tau_p\dot P_f=P_{PoC}-P_f,\qquad\tau_q\dot Q_f=Q_{PoC}-Q_f
$$

$$
\omega=1-m_p(P_f-P^*),\quad E=E_0-n_q(Q_f-Q^*),\quad\dot\delta=\omega_b(\omega-\omega_g)
$$

$$
I=\frac{Ee^{j\delta}-V}{Z_f},\qquad P+jQ=VI^*
$$

$$
e_v=v^*-v_c,\quad\dot\xi_v=e_v,\quad i_1^*=F i_2-\omega_b\omega cJv_c+K_{pv}e_v+K_{iv}\xi_v
$$

$$
e_i=i_1^*-i_1,\quad\dot\xi_i=e_i,\quad u=v_c-\omega_b\omega\ell_1Ji_1+K_{pi}e_i+K_{ii}\xi_i
$$

1. 运行 +0.03 pu P* 阶跃，观察频率并解释为何它先变化。
2. 检查匹配后的 Q*，与初始测量 Q 比较。
3. 施加 +0.1 Hz，比较最终 P 变化与 −0.1/(50 m_p)。


In [ ]:
case = {"mode": "droop", "scr": 5, "step": 0.03, "event": "p", "duration": 4}
result = solve(case)
print("states:", result["state_count"])
print("initial residual:", result["initial_residual"])
print("network / energy residual:", result["network_residual"])
print("reset:", result["reset_error"])
print("final:", {k: values[-1] for k, values in result["traces"].items()})


端口 Q = 0 时，这个 droop 模型的 Q* 也必须为零吗？

1. 是，对任何网络阻抗都如此
2. 只有 PLL 增益为零时如此
3. 不一定；Q* 可以提供支路所需的内部电压偏置

<details><summary>解释</summary>

Q_f = 0 时，平衡点满足 E = E₀ + n_q Q*。通过 Z_f 的电流要求特定 E，所以 Q* 用于匹配源电压幅值。端口目标与控制器指令是不同的量。

</details>


## C1-06 · VSM 与虚拟惯性

<h3>增加一个频率状态</h3>
<p>虚拟同步机用内部转速状态代替代数式 P–频率下垂。四状态教学模型包含 δ、ω、P_f、Q_f，电气源和无功规律与模块 5 相同，因此能够单独观察同步规律的影响。</p>
<p>使用 Mω̇ = P* − P_f − D(ω − 1)，M 的单位为秒，D = 1/m_p。在这套标幺 swing 约定下，若 M = 2H，则默认 M = 4 s 对应 H = 2 s。角度方程仍需用 ω_b 乘标幺频差。</p>
<h3>区分稳态斜率与惯性</h3>
<p>稳态满足 P_f = P* − D(ω_g − 1)。D = 1/m_p 时，它与 droop 控制器具有相同稳态斜率。M 改变暂态速度导数，不改变该稳态斜率。源仓库将调速器系数与显式阻尼分开，并约束两者之和为 1/m_p；本实验使用合并系数。</p>
<p>从原平衡点施加 +0.03 pu P* 后，初始 ω̇ = 0.03/4 = 0.0075 pu/s，即 0.375 Hz/s。保持其他初始条件和系数不变，M 加倍会使初始导数减半。</p>
<h3>电气功率由什么支撑？</h3>
<p>“虚拟惯性”描述一种控制规律。本实验假设理想直流源，以及不受限制的电压、电流实现，因此不能判定直流能量是否足够，也不能判定变流器过载能力。研究这些问题，需要加入直流储能动态、设备限制及其控制相互作用。</p>
<h3>不要将 GFL 虚拟惯性与 VSM 混为一谈</h3>
<p>GFL 虚拟惯性可以在保留 PLL 同步的基础上，增加因果滤波后的 RoCoF 功率请求。它与内部 VSM 转速状态具有不同结构。仓库 GFL-VI Notebook 保留了冻结参数下不稳定的案例；加入名称带“惯性”的项并不能保证稳定。</p>
<p>固定 SCR、下垂斜率和测量滤波，比较 M = 1、4、8 s，记录初始 RoCoF、功率峰值和收敛过程。低阶 VSM 的响应不能证明仓库高阶电气与内部控制实现稳定。</p><p>默认 4 s 窗口可能在 VSM 收敛前结束，尤其这里保留了 0.1 s 功率滤波。讨论稳态值前，可将时长延长至 20 s。某些参数组合可能出现增长的振荡；应保留该结果，并检查模型适用域。</p>

$$
M\dot\omega=P^*-P_f-D(\omega-1),\qquad D=\frac1{m_p},\qquad M=2H
$$

$$
\dot\delta=\omega_b(\omega-\omega_g),\qquad P_{\infty}=P^*-\frac{\omega_g-1}{m_p}
$$

$$
\dot f(1^+)=f_b\frac{\Delta P^*}{M}
$$

1. M = 4 s 时，预测 +0.03 pu 阶跃后的初始频率导数。
2. 固定 m_p、SCR，再分别用 M = 1、8 s 运行。
3. 用频率阶跃实验，对比 droop GFM 的稳态斜率。


In [ ]:
case = {"mode": "vsm", "scr": 5, "step": 0.03, "event": "p", "duration": 4}
result = solve(case)
print("states:", result["state_count"])
print("initial residual:", result["initial_residual"])
print("network / energy residual:", result["network_residual"])
print("reset:", result["reset_error"])
print("final:", {k: values[-1] for k, values in result["traces"].items()})


固定 D 和运行点，M 加倍后，同样 P* 阶跃对应的初始响应如何变化？

1. 初始频率导数减半
2. 稳态频率–功率斜率加倍
3. 不再需要直流能量来源

<details><summary>解释</summary>

初始时 Mω̇ = ΔP*，所以 ω̇ 与 M 成反比。平衡时 ω̇ = 0，M 从稳态功率–频率关系中消失。

</details>


## C1-07 · GFL–GFM 并联混合模型

<h3>用共享端口耦合两个支路</h3>
<p>实验将一个 GFL 电流执行支路和一个 droop-GFM 电压源支路连接到同一 PCC，两者同时工作，各分配总有功增量的一半。它们通过同一电网阻抗耦合；把两个分别仿真的单机曲线直接相加，会遗漏这种相互作用。</p>
<p>系统总基值保持为 10 kVA。两个支路都使用这个共同基值，各从 P = 0.3 pu、Q = 0 出发，总功率为 0.6 pu。若采用各自设备基值，应用共享网络方程前，必须换算电流、功率和阻抗。</p>
<h3>网络只求解一次</h3>
<p>电网阻抗为 Z_g，GFL 电流为 I_c，GFM 源 U = E exp(jδ) 通过 Z_f 接入。KCL 给出 I_grid = I_c + (U − V)/Z_f，再结合 V = V_g + Z_g I_grid，可得到下方显式 PCC 表达式。必须先求解它，再计算两个支路各自的功率反馈。</p>
<p>状态数为 4 + 3 = 7，但拼接状态还不够：两个控制器必须使用同一物理 PCC 电压，再分别转换到各自局部坐标。每个支路功率为 V Iₖ*，总功率为 V(I_c + I_v)*。</p>
<h3>支路功率不是任意加权的输出</h3>
<p>默认指令阶跃平均分配，但支路暂态不一定相同，因为控制动态不同。“支路功率”图展示 GFL、GFM 贡献和总量；任一时刻的支路功率之和，都必须等于 PCC 总功率。</p>
<h3>与 REGFM_C1 的关系</h3>
<p>这是<strong>共享 PCC 的并联教学模型</strong>，并非官方 REGFM_C1 实现。源 Notebook 明确区分正序 A11 表示与详细平均 F21 研究扩展。虚拟 R + jX 关系和物理 LCL 电路保留的动态不同。需要复现精确控制方程与参数时，应使用源 Notebook。</p>
<p>降低 SCR，同时检查支路功率与总功率。观察到的相互作用只对应这里的电流源/电压源表示、共同基值和功率分配策略，不能推导为对所有 hybrid 架构的排名。</p>

$$
I_g=I_c+\frac{U-V}{Z_f},\qquad V=V_g+Z_gI_g
$$

$$
V=\frac{V_g+Z_g I_c+(Z_g/Z_f)U}{1+Z_g/Z_f}
$$

$$
S_{PCC}=V(I_c+I_v)^*=S_c+S_v
$$

1. 选择支路功率，比较 +0.03 pu 总阶跃期间的两个贡献。
2. 检查事件时刻与最终时刻的支路功率之和是否等于总功率。
3. 将 SCR 降至 2 再运行，描述哪个支路响应变化更明显。


In [ ]:
case = {"mode": "parallel", "scr": 5, "step": 0.03, "event": "p", "duration": 4}
result = solve(case)
print("states:", result["state_count"])
print("initial residual:", result["initial_residual"])
print("network / energy residual:", result["network_residual"])
print("reset:", result["reset_error"])
print("final:", {k: values[-1] for k, values in result["traces"].items()})


为什么不能直接相加两个独立仿真的支路轨迹？

1. 同一端口的功率不能相加
2. 会遗漏共同 PCC 电压反馈及网络耦合
3. 并联支路必须具有相同状态

<details><summary>解释</summary>

在同一 PCC 计算时，功率可以相加。独立仿真遗漏的是共享电压：每个支路都会改变它，从而改变另一支路的反馈。

</details>


## C1-08 · 模式切换混合模型

<h3>模式决定当前哪些方程生效</h3>
<p>模式切换逆变器每次只有一个控制器处于激活状态。令 m ∈ {GFL, VSM}，事件之间满足 ẋ = f_m(x,u)，切换时满足 x⁺ = R(x⁻,u)。活动状态向量与用于保存变量的联合 checkpoint 是两个概念，联合保存表不能直接视为更大的 ODE。</p>
<p>这个演示先使用四状态 GFL，在 1 s 施加指令扰动，在 2 s 切换到四状态 VSM。虽然两者长度都是四，但含义不同：[δ_PLL, ξ_PLL, i_d, i_q] 不能原样复制为 [δ_source, ω, P_f, Q_f]。</p>
<h3>构造满足端口关系的重置</h3>
<p>切换前记录 PCC 电压 V、注入电流 I、P、Q 和 PLL 频率 ω。接入的新源必须满足 U⁺ = V⁻ + Z_f I⁻，角度设为 arg(U⁺)，频率设为 ω⁻，功率滤波状态设为 P⁻、Q⁻。通过新源阻抗，这些条件重建相同的端口电压和电流。</p>
<p>再设 E₀⁺ = |U⁺| + n_q(Q⁻ − Q*)，P*⁺ = P⁻ + D(ω⁻ − 1)，使电压偏置匹配、初始 VSM 加速度为零。这是<strong>重新校准接入控制器指令和电压偏置</strong>的显式策略，并不保证切换前 P* 原封不动地保留。</p>
<h3>检查哪些量连续</h3>
<p>重置检查取复数 PCC 电压、P、Q、内部频率跳变量的最大值；默认结果应接近浮点舍入误差。源角度允许变化，因为接入源位于阻抗之后，而退出角度属于 PLL。这里不应把角度数值相等作为连续性条件。</p>
<p>1 s 的独立扰动仍可能引起代数电压跳变。重置条件针对 2 s 的切换事件，不意味着任意输入变化都保持连续。</p>
<h3>详细切换模型还需要什么</h3>
<p>实验只包含一次定时切换，未加入非活动控制器跟踪、限幅、驻留时间或保护触发。仓库 VSM–PLL 切换 Notebook 进一步处理 hold/release 策略和不同活动状态维度，也区分了稳定的控制器专用演示参数与相同增益下不稳定的 GFM 端点。仅验证两个端点，不能验证同一设备的模式切换。</p>

$$
\dot x=f_m(x,u),\qquad x^+=R_{m^-\to m^+}(x^-,u)
$$

$$
U^+=V^-+Z_f I^-,\quad\delta^+=\arg U^+,\quad\omega^+=\omega^-,\quad P_f^+=P^-,\quad Q_f^+=Q^-
$$

$$
E_0^+=|U^+|+n_q(Q^--Q^*),\qquad P^{*+}=P^-+D(\omega^--1)
$$

1. 运行定时切换，检查 2 s 前后的轨迹。
2. 读取数值检查中的重置残差和最终指令偏置。
3. 在 reset_to_vsm() 中，故意将重置角度改为 PLL 角度，观察跳变。


In [ ]:
case = {"mode": "switch", "scr": 5, "step": 0.03, "event": "p", "duration": 4}
result = solve(case)
print("states:", result["state_count"])
print("initial residual:", result["initial_residual"])
print("network / energy residual:", result["network_residual"])
print("reset:", result["reset_error"])
print("final:", {k: values[-1] for k, values in result["traces"].items()})


GFL 与 VSM 状态向量长度相同时，可以直接复制吗？

1. 可以，长度相同就兼容
2. 可以，只要都使用标幺值
3. 不可以；需要按物理含义和端口条件映射

<details><summary>解释</summary>

GFL 的积分量、电流不是 VSM 的速度、滤波功率。应从 V、I 重构接入源，再映射频率和功率测量。重置由状态含义决定，而非向量长度。

</details>


## C1-09 · 平衡点、扰动与公平比较

<h3>在同一物理端口规定目标</h3>
<p>公平比较先统一 PoC 的 P、Q、电压、频率、系统基值和电网条件；实现这些目标所需的控制指令可以不同。实验中，GFL、droop、VSM、parallel 的总 P 均为 0.6 pu、Q 均为零；parallel 将端口目标平均分给两个支路。</p>
<p>基值为 S_b = 10 kVA、V_b = 400 V、f_b = 50 Hz。初始电网幅值 1 pu，X/R = 10，|Z_g| = 1/SCR。这里的 SCR 基于共同系统基值；并联支路额定容量不会悄悄改变电网阻抗。</p>
<h3>先求平衡点，再加扰动</h3>
<p>端口满足 S = VI*、V = 1 + Z_g I。消去 I 后，得到 V = 1 + Z_g conj(S/V)。选择高电压根，再分别在各控制器的坐标中初始化，并检查 max|f(x₀,u₀)|。Droop、VSM 的 Q* 偏置用于补偿源阻抗电压降；强制原始 Q* 相同，会得到不同的端口运行点。</p>
<h3>保持可复现的事件协议</h3>
<p>先以旧输入积分 0–1 s，再以新输入积分 1–4 s。求解器将事件精确放在积分边界，1 s 的相邻采样显示事件前后值，避免 Runge–Kutta 子步跨越不连续点而平均输入。切换实验还在 2 s 结束一段积分，并应用显式重置。</p>
<p>默认积分步长为 0.5 ms。可在 Python 实验中将步长减半并比较轨迹；曲线平滑并不能证明数值精度足够。</p>
<h3>区分三类证据</h3>
<div class="ibr-table"><table><thead><tr><th>证据类别</th><th>检查方式</th><th>能够证明什么</th></tr></thead><tbody><tr><td>方程一致性</td><td>平衡点、KCL、功率求和、重置残差</td><td>所声明的方程与接口一致</td></tr><tr><td>模型有效性</td><td>在声明范围内与显式高阶实现比较</td><td>该研究范围内的近似误差有界</td></tr><tr><td>外部验证</td><td>独立测量或合格参考模型</td><td>与该外部证据的符合程度</td></tr></tbody></table></div>
<p>网页实验完成的是第一类检查。源仓库六类别比较 Notebook 显式保留稳定与不稳定条目，记录共同运行点协议。不要为叠加出漂亮曲线而删掉不稳定模型，也不要从一次阶跃实验推导所有控制类别的全局排名。</p>
<h3>结课项目</h3>
<p>提交模型约定、参数快照和 Python 实验、匹配平衡点表、两类扰动比较，以及响应差异解释。说明数值步长细化结果、保留输出和适用范围。进阶项目可将同一协议用于仓库完整阶次 Notebook，再说明教学模型中的结论如何变化。</p><p>默认 4 s 窗口可能在 VSM 收敛前结束，尤其这里保留了 0.1 s 功率滤波。讨论稳态值前，可将时长延长至 20 s。某些参数组合可能出现增长的振荡；应保留该结果，并检查模型适用域。</p>

$$
S=VI^*,\qquad V=1+Z_g\overline{S/V},\qquad\lVert f(x_0,u_0)\rVert_\infty<\varepsilon
$$

$$
Z_g=\frac{1+j\,10}{\mathrm{SCR}\sqrt{101}},\qquad\Delta P^*_{branch}=\frac{\Delta P^*_{total}}{N}
$$

$$
e_h=\max_k|P_h(t_k)-P_{h/2}(t_k)|
$$

1. 在 SCR = 5 下，用总 +0.03 pu 有功阶跃比较四个模型。
2. 改为 +0.1 Hz 频率阶跃，解释 GFL 与 GFM 的稳态响应。
3. 在 Python 中将 case["dt"] 减半，对齐采样后比较，并在报告中记录最大误差。


In [ ]:
case = {"mode": "compare", "scr": 5, "step": 0.03, "event": "p", "duration": 4}
result = solve(case)
print("states:", result["state_count"])
print("initial residual:", result["initial_residual"])
print("network / energy residual:", result["network_residual"])
print("reset:", result["reset_error"])
print("final:", {k: values[-1] for k, values in result["traces"].items()})


比较不同控制类别之前，应优先匹配什么？

1. 端口运行点、基值和扰动协议
2. 每一个原始控制指令
3. 只匹配状态数

<details><summary>解释</summary>

不同控制规律可能需要不同内部指令，才能产生同一端口 P、Q。应统一声明的端口条件，并保留各模型自己的状态初始化。原始指令相同不能单独保证公平。

</details>
